In [ ]:
!pip install chromadb
import os
import re
import json
import uuid
import shutil

from typing import TypedDict, List

from sentence_transformers import SentenceTransformer

import chromadb

from langgraph.graph import StateGraph, START, END

from pydantic import BaseModel, Field

print("Libraries imported successfully.")

# ============================================================

# SECTION 3 — CREATE PROJECT STRUCTURE

# ============================================================

os.makedirs("support_assistant", exist_ok=True)

os.makedirs(
    "support_assistant/policy_docs",
    exist_ok=True
)

os.makedirs(
    "support_assistant/chroma_db",
    exist_ok=True
)

print("Project folders created.")

# ============================================================

# SECTION 4 — MOCK LLM CONFIGURATION

# ============================================================

# MOCK_LLM=1 is the default graded configuration.
#
# This means:
# - No external LLM API
# - No network call
# - Deterministic responses
# - Retrieval is based entirely on local embeddings
os.environ["MOCK_LLM"] = "1"

MOCK_LLM = os.getenv(
    "MOCK_LLM",
    "1"
)

print("MOCK_LLM =", MOCK_LLM)

# ============================================================
# ============================================================

# SECTION 5 — EXACT POLICY DOCUMENTS FROM CAPSTONE

# ============================================================

policy_documents = {

    "doc_01_delivery_policy.txt": """
Zepto delivers grocery and household essentials to serviceable
pin codes within 10 to 30 minutes of order confirmation,
depending on the customer's delivery zone and current order
volume.

Standard delivery is free on orders over INR 149; orders below
this threshold incur a flat INR 25 delivery fee.

Priority delivery, which reserves the next available rider slot,
is available at checkout for an additional INR 15.

Zepto does not currently deliver to addresses outside its listed
serviceable pin codes.
""",

    "doc_02_returns_refunds.txt": """
Grocery and perishable items may be reported for a return within
24 hours of delivery if damaged, spoiled, or incorrect;
nony-perishable packaged items may be returned within 7 days of
delivery in unopened, resalable condition.

Approved refunds are credited to the original payment method
within 3–5 business days, or instantly to the Zepto wallet if the
customer opts for wallet credit.

Personal care items that have been opened are non-returnable
except in the case of a manufacturing defect.

Return pickup, where required, is arranged free of cost by Zepto.
""",

    "doc_03_membership_tiers.txt": """
Zepto offers three account tiers:

Basic (free, default tier, standard delivery fees apply).

Zepto Pass (INR 49 per month, free standard delivery on all
orders and 5% off select categories).

Zepto Pass+ (INR 99 per month, free priority delivery, 10% off
select categories, and early access to limited-time deals
24 hours before they go live to Basic and Pass members).

Membership can be cancelled at any time from account settings;
cancelling stops the next billing cycle but does not refund the
current membership period.
""",

    "doc_04_order_tracking.txt": """
Every Zepto order shows a live rider-tracking map from the moment
it is packed until delivery, accessible from the 'Track Order'
screen.

Estimated delivery time updates automatically as the rider moves.

If an order's status shows no movement for more than 20 minutes
past its original estimated delivery time, customers should
contact support directly rather than continue waiting, since this
indicates a likely delivery issue.
""",

    "doc_05_order_cancellation_policy.txt": """
Orders can be cancelled free of cost any time before the order
status changes to 'Packed', typically within the first 2 minutes
of placing the order.

Once an order has been packed, it can no longer be cancelled
through the app, since the rider is dispatched immediately after
packing given Zepto's quick-delivery model.

If a packed order cannot be delivered due to a Zepto-side issue
(for example, rider unavailability), the order is auto-cancelled
and fully refunded without any cancellation fee.
""",

    "doc_06_damaged_or_missing_items.txt": """
If an order arrives with damaged, spoiled, or missing items,
customers must report it within 24 hours of delivery through the
'Report an Issue' button on the order page.

Zepto ships a free replacement or issues a full refund for
damaged, spoiled, or missing items without requiring the customer
to return the original item, unless the order value exceeds
INR 1000, in which case a photo of the issue must be submitted
through the report form before a replacement or refund is
processed.
""",

    "doc_07_gift_cards.txt": """
Zepto gift cards are available in fixed denominations of
INR 100, INR 250, INR 500, and INR 1000, and are delivered by
email or SMS within minutes of purchase.

Gift cards are valid for 1 year from the date of issue and carry
no maintenance fees.

Gift card balance can be combined with one other payment method
at checkout but cannot be combined with another gift card in the
same transaction.

Gift card balance cannot be redeemed for cash except where
required by law.
""",

    "doc_08_customer_support_hours.txt": """
Zepto customer support is available via in-app chat 24 hours a
day, 7 days a week, given the time-sensitive nature of quick
commerce deliveries.

Average in-app chat response time is under 2 minutes.

Email support is also available for non-urgent queries and is
answered within 24 hours on business days.

Phone support is not offered.
"""
}

print(
    "Created",
    len(policy_documents),
    "exact policy documents."
)

# Clear existing policy documents from the directory
policy_docs_dir = "support_assistant/policy_docs"
for filename in os.listdir(policy_docs_dir):
    filepath = os.path.join(policy_docs_dir, filename)
    if os.path.isfile(filepath):
        os.remove(filepath)
print("Cleared existing policy documents.")

for filename in policy_documents:

    # Write policy documents to files
    filepath = os.path.join(
        "support_assistant",
        "policy_docs",
        filename
    )
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(policy_documents[filename].strip())
    print(filename)

# ============================================================

# SECTION 6 — READ POLICY DOCUMENTS

# ============================================================

documents = []

for filename in sorted(
    os.listdir("support_assistant/policy_docs")
):

    filepath = os.path.join(
        "support_assistant",
        "policy_docs",
        filename
    )

    with open(
        filepath,
        "r",
        encoding="utf-8"
    ) as f:

        text = f.read().strip()

    documents.append({
        "document_id": filename,
        "text": text
    })

print("Documents loaded:")

for doc in documents:
    print(
        doc["document_id"],
        "->",
        len(doc["text"]),
        "characters"
    )

# ============================================================

# SECTION 7 — CHUNK DOCUMENTS

# ============================================================

# Simple paragraph-based chunking.
#
# Each paragraph becomes a searchable chunk.
# This keeps the implementation deterministic and easy to
# inspect.

chunks = []

for doc in documents:

    paragraphs = [
        p.strip()
        for p in doc["text"].split("\n\n")
        if p.strip()
    ]

    for index, paragraph in enumerate(paragraphs):

        chunk_id = (
            doc["document_id"]
            .replace(".txt", "")
            + "_chunk_"
            + str(index)
        )

        chunks.append({
            "chunk_id": chunk_id,
            "document_id": doc["document_id"],
            "text": paragraph
        })

print("Total chunks:", len(chunks))

# ============================================================

# SECTION 8 — LOAD SENTENCE TRANSFORMER MODEL

# ============================================================

# Required local embedding model:
# all-MiniLM-L6-v2

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

# ============================================================

# SECTION 9 — GENERATE EMBEDDINGS

# ============================================================

chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True
)

print(
    "Generated embeddings:",
    embeddings.shape
)

# ============================================================

# SECTION 10 — CREATE CHROMADB COLLECTION

# ============================================================

chroma_client = chromadb.PersistentClient(
    path="support_assistant/chroma_db"
)

# Delete existing collection if this notebook cell is rerun

try:
    chroma_client.delete_collection(
        name="policy_collection"
    )
except Exception:
    pass

collection = chroma_client.create_collection(
    name="policy_collection",
    metadata={
        "description": "Local support policy knowledge base"
    }
)

# ============================================================

# SECTION 11 — STORE EMBEDDINGS IN CHROMADB

# ============================================================

collection.add(
    ids=[
        chunk["chunk_id"]
        for chunk in chunks
    ],
    embeddings=[
        embedding.tolist()
        for embedding in embeddings
    ],
    documents=chunk_texts,
    metadatas=[
        {
            "document_id": chunk["document_id"]
        }
        for chunk in chunks
    ]
)

print(
    "Documents stored in ChromaDB:",
    collection.count()
)

# ============================================================

# SECTION 12 — TEST RETRIEVAL

# ============================================================

def retrieve_documents(
    query,
    top_k=3
):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k
    )

    retrieved = []

    for i in range(
        len(results["ids"][0])
    ):

        retrieved.append({
            "chunk_id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "document_id": results["metadatas"][0][i][
                "document_id"
            ],
            "distance": results["distances"][0][i]
            if "distances" in results
            else None
        })

    return retrieved

test_query = "How can I track my order?"

retrieved = retrieve_documents(
    test_query,
    top_k=3
)

print("Query:", test_query)
print("\nRetrieved documents:")

for item in retrieved:

    print("\nChunk ID:", item["chunk_id"])
    print("Document:", item["document_id"])
    print("Text:", item["text"])

# ============================================================

# SECTION 13 — INTENT CLASSIFICATION

# ============================================================

# Required heuristic:
#
# if lowercase query contains one of:
# delivery, return, refund, membership, tracking,
# cancel, gift card, support hours
#
# => policy_question
#
# Otherwise:
# => general_question

POLICY_KEYWORDS = [
    "delivery",
    "return",
    "refund",
    "membership",
    "tracking",
    "track", # Added 'track' to handle variations of the word
    "cancel",
    "gift card",
    "support hours"
]

def classify_query(
    query: str
) -> str:

    query_lower = query.lower()

    for keyword in POLICY_KEYWORDS:

        if keyword in query_lower:
            return "policy_question"

    return "general_question"

# Test classifier

test_queries = [
    "How can I track my order?",
    "What is the refund policy?",
    "Hello, how are you?",
    "Tell me a joke"
]

for query in test_queries:

    print(
        query,
        "->",
        classify_query(query)
    )

# ============================================================

# SECTION 14 — LANGGRAPH STATE

# ============================================================

class SupportState(TypedDict):

    query: str
    intent: str
    retrieved_chunks: list
    answer: str
    sources: list
    confidence: float

# ============================================================

# SECTION 15 — REQUIRED NODE 1

# classify_intent

# ============================================================

def classify_intent(
    state: SupportState
):

    intent = classify_query(
        state["query"]
    )

    return {
        "intent": intent
    }

# ============================================================

# SECTION 16 — REQUIRED NODE 2

# retrieve_and_answer

# ============================================================

def retrieve_and_answer(
    state: SupportState
):

    query = state["query"]

    retrieved = retrieve_documents(
        query,
        top_k=3
    )

    if len(retrieved) > 0:

        top_chunk = retrieved[0]["text"]

        # Required deterministic mock answer
        answer = (
            "Based on the retrieved context: "
            + top_chunk
        )

    else:

        answer = (
            "I could not find relevant policy information."
        )


    source_ids = [
        item["chunk_id"]
        for item in retrieved
    ]


    # Deterministic confidence for mock mode.
    #
    # This is deliberately simple and reproducible.
    if retrieved:
        confidence = 0.90
    else:
        confidence = 0.20


    return {
        "retrieved_chunks": retrieved,
        "answer": answer,
        "sources": source_ids,
        "confidence": confidence
    }

# ============================================================

# SECTION 17 — REQUIRED NODE 3

# direct_answer

# ============================================================

def direct_answer(
    state: SupportState
):

    answer = (
        "I can only answer questions about "
        "Zepto policies right now."
    )

    return {
        "answer": answer,
        "sources": [],
        "confidence": 1.0
    }

# ============================================================

# SECTION 18 — CONDITIONAL ROUTING

# ============================================================

def route_by_intent(
    state: SupportState
):

    if state["intent"] == "policy_question":

        return "retrieve_and_answer"

    return "direct_answer"

# ============================================================

# SECTION 19 — BUILD LANGGRAPH

# ============================================================

graph_builder = StateGraph(
    SupportState
)

# Add required named nodes

graph_builder.add_node(
    "classify_intent",
    classify_intent
)

graph_builder.add_node(
    "retrieve_and_answer",
    retrieve_and_answer
)

graph_builder.add_node(
    "direct_answer",
    direct_answer
)

# Start → classify

graph_builder.add_edge(
    START,
    "classify_intent"
)

# Conditional routing

graph_builder.add_conditional_edges(
    "classify_intent",
    route_by_intent,
    {
        "retrieve_and_answer":
            "retrieve_and_answer",

        "direct_answer":
            "direct_answer"
    }

)

# Both branches → END

graph_builder.add_edge(
    "retrieve_and_answer",
    END
)

graph_builder.add_edge(
    "direct_answer",
    END
)

support_graph = graph_builder.compile()

print("LangGraph compiled successfully.")

# ============================================================

# SECTION 20 — PYDANTIC FINAL RESPONSE SCHEMA

# ============================================================

class SupportResponse(BaseModel):

    answer: str

    sources: List[str]

    confidence: float = Field(
        ge=0.0,
        le=1.0
    )

print("Pydantic response schema created.")

# ============================================================

# SECTION 21 — ASSISTANT FUNCTION

# ============================================================

def ask_assistant(
    query: str
):

    initial_state: SupportState = {
        "query": query,
        "intent": "",
        "retrieved_chunks": [],
        "answer": "",
        "sources": [],
        "confidence": 0.0
    }


    result = support_graph.invoke(
        initial_state
    )


    # Validate final result
    validated = SupportResponse(
        answer=result["answer"],
        sources=result.get(
            "sources",
            []
        ),
        confidence=result.get(
            "confidence",
            0.0
        )
    )


    return validated

# ============================================================

# SECTION 22 — TEST POLICY QUESTION

# ============================================================

policy_question = (
    "How can I track my order?"
)

response_1 = ask_assistant(
    policy_question
)

print(
    response_1.model_dump_json(
        indent=2
    )
)

# ============================================================

# SECTION 23 — TEST GENERAL QUESTION

# ============================================================

general_question = (
    "What is the capital of India?"
)

response_2 = ask_assistant(
    general_question
)

print(
    response_2.model_dump_json(
        indent=2
    )
)

# ============================================================

# SECTION 24 — REQUIRED EXAMPLE JSON OUTPUTS

# ============================================================

example_outputs = {
    "retrieval_example": response_1.model_dump(),
    "general_example": response_2.model_dump()
}

with open(
    "support_assistant/example_outputs.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        example_outputs,
        f,
        indent=2
    )

print(
    json.dumps(
        example_outputs,
        indent=2
    )
)

# ============================================================

# SECTION 25 — FASTAPI APPLICATION

# ============================================================

from fastapi import FastAPI

app = FastAPI(
    title="Zepto Policy Support Assistant",
    version="1.0.0"
)

class AskRequest(BaseModel):

    query: str

@app.post(
    "/ask",
    response_model=SupportResponse
)
def ask_endpoint(
    request: AskRequest
):

    return ask_assistant(
        request.query
    )

print("FastAPI application created.")

# ============================================================

# SECTION 26 — SAVE FASTAPI APP

# ============================================================

fastapi_code = r'''
import os
from typing import TypedDict, List

import chromadb

from sentence_transformers import SentenceTransformer

from langgraph.graph import StateGraph, START, END

from pydantic import BaseModel, Field

from fastapi import FastAPI

os.environ["MOCK_LLM"] = os.getenv(
    "MOCK_LLM",
    "1"
)

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

chroma_client = chromadb.PersistentClient(
    path="support_assistant/chroma_db"
)

collection = chroma_client.get_collection(
    name="policy_collection"
)

POLICY_KEYWORDS = [
    "delivery",
    "return",
    "refund",
    "membership",
    "tracking",
    "track",
    "cancel",
    "gift card",
    "support hours"
]

def retrieve_documents(
    query,
    top_k=3
):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k
    )

    retrieved = []

    for i in range(
        len(results["ids"][0])
    ):

        retrieved.append({
            "chunk_id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "document_id": results["metadatas"][0][i][
                "document_id"
            ]
        })

    return retrieved

def classify_query(query: str):

    query_lower = query.lower()

    for keyword in POLICY_KEYWORDS:

        if keyword in query_lower:
            return "policy_question"

    return "general_question"

class SupportState(TypedDict):

    query: str
    intent: str
    retrieved_chunks: list
    answer: str
    sources: list
    confidence: float

def classify_intent(state):

    return {
        "intent": classify_query(
            state["query"]
        )
    }

def retrieve_and_answer(state):

    retrieved = retrieve_documents(
        state["query"],
        top_k=3
    )

    if retrieved:

        answer = (
            "Based on the retrieved context: "
            + retrieved[0]["text"]
        )

        confidence = 0.90

    else:

        answer = (
            "I could not find relevant policy information."
        )

        confidence = 0.20


    return {
        "retrieved_chunks": retrieved,
        "answer": answer,
        "sources": [
            item["chunk_id"]
            for item in retrieved
        ],
        "confidence": confidence
    }

def direct_answer(state):

    return {
        "answer":
            "I can only answer questions about "
            "Zepto policies right now.",

        "sources": [],

        "confidence": 1.0
    }

def route_by_intent(state):

    if state["intent"] == "policy_question":

        return "retrieve_and_answer"

    return "direct_answer"

graph_builder = StateGraph(
    SupportState
)

graph_builder.add_node(
    "classify_intent",
    classify_intent
)

graph_builder.add_node(
    "retrieve_and_answer",
    retrieve_and_answer
)

graph_builder.add_node(
    "direct_answer",
    direct_answer
)

graph_builder.add_edge(
    START,
    "classify_intent"
)

graph_builder.add_conditional_edges(
    "classify_intent",
    route_by_intent,
    {
        "retrieve_and_answer":
            "retrieve_and_answer",

        "direct_answer":
            "direct_answer"
    }

)

graph_builder.add_edge(
    "retrieve_and_answer",
    END
)

graph_builder.add_edge(
    "direct_answer",
    END
)

support_graph = graph_builder.compile()

class SupportResponse(BaseModel):

    answer: str

    sources: List[str]

    confidence: float = Field(
        ge=0.0,
        le=1.0
    )

def ask_assistant(query):

    state = {
        "query": query,
        "intent": "",
        "retrieved_chunks": [],
        "answer": "",
        "sources": [],
        "confidence": 0.0
    }

    result = support_graph.invoke(
        state
    )

    return SupportResponse(
        answer=result["answer"],
        sources=result.get(
            "sources",
            []
        ),
        confidence=result.get(
            "confidence",
            0.0
        )
    )

app = FastAPI(
    title="Zepto Policy Support Assistant",
    version="1.0.0"
)

class AskRequest(BaseModel):

    query: str

@app.post(
    "/ask",
    response_model=SupportResponse
)
def ask_endpoint(request: AskRequest):

    return ask_assistant(
        request.query
    )

'''

with open(
    "support_assistant/app.py",
    "w",
    encoding="utf-8"
) as f:

    f.write(fastapi_code)

print(
    "FastAPI application saved to:",
    "support_assistant/app.py"
)

# ============================================================

# SECTION 27 — REQUIREMENTS.TXT

# ============================================================

requirements_text = """
sentence-transformers
chromadb
langgraph
fastapi
uvicorn
pydantic
"""

with open(
    "support_assistant/requirements.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        requirements_text.strip()
    )

print("requirements.txt created.")

# ============================================================

# SECTION 28 — DOCKERFILE

# ============================================================

dockerfile_text = """
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

ENV MOCK_LLM=1

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

with open(
    "support_assistant/Dockerfile",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        dockerfile_text.strip()
    )

print("Dockerfile created.")

# ============================================================

# SECTION 29 — CREATE README FOR MODULE 3

# ============================================================

module3_readme = """

# Module 3 — Support Assistant

## Architecture

Policy documents
→ text chunking
→ Sentence Transformer embeddings
→ ChromaDB
→ query embedding
→ top-3 cosine-similarity retrieval
→ LangGraph routing
→ Pydantic validation
→ FastAPI `/ask`

## Embedding model

`all-MiniLM-L6-v2`

The embedding model runs locally.

## LangGraph nodes

The graph contains three required named nodes:

1. `classify_intent`
2. `retrieve_and_answer`
3. `direct_answer`

## Intent classification

A query is classified as `policy_question` when its lowercase
text contains one of:

* delivery
* return
* refund
* membership
* tracking
* cancel
* gift card
* support hours

All other queries are classified as `general_question`.

## Mock LLM

`MOCK_LLM=1` is the default configuration.

In mock mode, policy answers are deterministic:

`Based on the retrieved context: {top_chunk_snippet}`

General questions return:

`I can only answer questions about Zepto policies right now.`

No external LLM API is required.

## FastAPI

Start the API with:

```bash
uvicorn app:app --host 0.0.0.0 --port 8000
```

Endpoint:

`POST /ask`

Example request:

```json
{
  "query": "How can I track my order?"
}
```

## Docker

Build:

```bash
docker build -t zepto-support-assistant .
```

Run:

```bash
docker run -p 8000:8000 zepto-support-assistant
```

"""

with open(
    "support_assistant/README.md",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        module3_readme.strip()
    )

print("Module 3 README created.")

# ============================================================

# SECTION 30 — LIST GENERATED FILES

# ============================================================

print("\nModule 3 files:")

for root, dirs, files in os.walk(
    "support_assistant"
):

    for filename in files:

        filepath = os.path.join(
            root,
            filename
        )

        print(filepath)

# ============================================================

# SECTION 31 — FINAL MODULE 3 TEST

# ============================================================

print("\n" + "=" * 60)
print("MODULE 3 FINAL TEST")
print("=" * 60)

test_cases = [
    "How can I track my order?",
    "What is the refund policy?",
    "Can I cancel my order?",
    "What is the capital of India?"
]

for query in test_cases:

    result = ask_assistant(query)

    print("\nQuery:", query)
    print(
        result.model_dump_json(
            indent=2
        )
    )

# print("""

# MODULE 3 IMPLEMENTATION COMPLETE

# - 8 policy documents
# - Local sentence-transformer embeddings
# - all-MiniLM-L6-v2
# - ChromaDB vector database
# - Top-3 retrieval
# - Cosine-similarity retrieval
# - MOCK_LLM=1
# - classify_intent node
# - retrieve_and_answer node
# - direct_answer node
# - Conditional LangGraph routing
# - TypedDict state
# - Pydantic final response schema
# - Deterministic mock response
# - Source IDs
# - Confidence score
# - FastAPI /ask endpoint
# - Example JSON outputs
# - requirements.txt
# - Dockerfile
# - README

# ============================================================
# ")

Libraries imported successfully.
Project folders created.
MOCK_LLM = 1
Created 8 exact policy documents.
Cleared existing policy documents.
doc_01_delivery_policy.txt
doc_02_returns_refunds.txt
doc_03_membership_tiers.txt
doc_04_order_tracking.txt
doc_05_order_cancellation_policy.txt
doc_06_damaged_or_missing_items.txt
doc_07_gift_cards.txt
doc_08_customer_support_hours.txt
Documents loaded:
doc_01_delivery_policy.txt -> 506 characters
doc_02_returns_refunds.txt -> 563 characters
doc_03_membership_tiers.txt -> 525 characters
doc_04_order_tracking.txt -> 429 characters
doc_05_order_cancellation_policy.txt -> 499 characters
doc_06_damaged_or_missing_items.txt -> 487 characters
doc_07_gift_cards.txt -> 471 characters
doc_08_customer_support_hours.txt -> 336 characters
Total chunks: 29


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.
Generated embeddings: (29, 384)
Documents stored in ChromaDB: 29
Query: How can I track my order?

Retrieved documents:

Chunk ID: doc_04_order_tracking_chunk_0
Document: doc_04_order_tracking.txt
Text: Every Zepto order shows a live rider-tracking map from the moment
it is packed until delivery, accessible from the 'Track Order'
screen.

Chunk ID: doc_06_damaged_or_missing_items_chunk_0
Document: doc_06_damaged_or_missing_items.txt
Text: If an order arrives with damaged, spoiled, or missing items,
customers must report it within 24 hours of delivery through the
'Report an Issue' button on the order page.

Chunk ID: doc_04_order_tracking_chunk_2
Document: doc_04_order_tracking.txt
Text: If an order's status shows no movement for more than 20 minutes
past its original estimated delivery time, customers should
contact support directly rather than continue waiting, since this
indicates a likely delivery issue.
How can I track my order? -> policy_questi

In [ ]:
test_questions = [
    "How much is standard delivery?",
    "How long do I have to report a damaged grocery item?",
    "How much does Zepto Pass cost?",
    "How can I track my order?",
    "Can I cancel my order after it is packed?",
    "What happens if an item is missing?",
    "How long are gift cards valid?",
    "Is customer support available 24 hours?"

]

for question in test_questions:

    result = ask_assistant(question)

    print("\nQuestion:", question)
    print(result.model_dump_json(indent=2))


Question: How much is standard delivery?
{
  "answer": "Based on the retrieved context: Standard delivery is free on orders over INR 149; orders below\nthis threshold incur a flat INR 25 delivery fee.",
  "sources": [
    "doc_01_delivery_policy_chunk_1",
    "doc_03_membership_tiers_chunk_1",
    "doc_03_membership_tiers_chunk_2"
  ],
  "confidence": 0.9
}

Question: How long do I have to report a damaged grocery item?
{
  "answer": "I can only answer questions about Zepto policies right now.",
  "sources": [],
  "confidence": 1.0
}

Question: How much does Zepto Pass cost?
{
  "answer": "I can only answer questions about Zepto policies right now.",
  "sources": [],
  "confidence": 1.0
}

Question: How can I track my order?
{
  "answer": "Based on the retrieved context: Every Zepto order shows a live rider-tracking map from the moment\nit is packed until delivery, accessible from the 'Track Order'\nscreen.",
  "sources": [
    "doc_04_order_tracking_chunk_0",
    "doc_06_damaged_or_m

# **Test FastAPI**

In [ ]:
import os

print(os.path.exists("support_assistant/app.py"))

True


In [ ]:
print(os.listdir("support_assistant"))

['app.py', 'policy_docs', 'Dockerfile', '__pycache__', 'requirements.txt', 'README.md', 'chroma_db', 'example_outputs.json']


In [ ]:
import subprocess
import time
import os
import requests # Added import

# Terminate any existing process if it's still running from a previous execution
# This logic attempts to clean up processes started by this specific cell's variable 'process'.
# It might not catch all zombie processes if the kernel was restarted or if processes were started externally.
if 'process' in locals() and process.poll() is None:
    print("Terminating existing FastAPI server process initiated by this cell...")
    process.terminate()
    try:
        # Give it a moment to terminate
        process.wait(timeout=5)
        print("Existing process terminated.")
    except subprocess.TimeoutExpired:
        print("Existing process did not terminate gracefully, killing it...")
        process.kill()

# Clear the process variable to ensure we don't accidentally refer to a stale one
if 'process' in locals():
    del process

# Define the URL for the FastAPI server
url = "http://127.0.0.1:8000/ask" # Define url here for consistent access
server_base_url = "http://127.0.0.1:8000" # Base URL for health checks

# Start the uvicorn process, capturing stdout and stderr for debugging purposes.
# This helps diagnose startup issues that might be hidden by DEVNULL.
process = subprocess.Popen(
    [
        "uvicorn",
        "support_assistant.app:app",
        "--host",
        "0.0.0.0", # Changed host to 0.0.0.0 for broader access in Colab
        "--port",
        "8000"
    ],
    stdout=subprocess.PIPE, # Capture stdout for debugging
    stderr=subprocess.PIPE  # Capture stderr for debugging
)

print("Attempting to start FastAPI server...")

# Health check loop to ensure the server is truly responsive
server_ready = False
max_startup_retries = 15 # Increased retries for Colab environment
startup_retry_delay_seconds = 2 # Check every 2 seconds

for i in range(max_startup_retries):
    time.sleep(startup_retry_delay_seconds)
    if process.poll() is not None: # Check if process has already terminated
        print(f"ERROR: FastAPI server process terminated prematurely during startup (attempt {i+1}/{max_startup_retries}).")
        stdout_data, stderr_data = process.communicate()
        print("Server stdout:\n", stdout_data.decode())
        print("Server stderr:\n", stderr_data.decode())
        server_ready = False
        break # Exit loop as process is dead

    try:
        # Try to connect to a known endpoint, like /docs or the root
        response = requests.get(f"{server_base_url}/docs", timeout=1)
        if response.status_code == 200:
            server_ready = True
            print("FastAPI server is responsive.")
            break
    except requests.exceptions.ConnectionError:
        pass # Server not ready yet, continue retrying
    except Exception as e:
        print(f"An unexpected error occurred during health check: {e}")
        server_ready = False
        break

if server_ready:
    print("FastAPI server started successfully on port 8000.")
    # If the process is still running (poll() is None) and responsive,
    # we don't call communicate() as that would close the pipes and prevent
    # the process from running in the background. We can leave it running.
else:
    print("ERROR: FastAPI server did not become responsive within the expected time.")
    print("Please check the output above for any error messages from the server.")
    if process.poll() is None: # If process is still running but not responsive
        print("The server process appears to be running, but is not responding to requests.")
        print("You might need to manually inspect its logs or terminate it.")
        # Attempt to terminate to avoid zombie processes
        print("Attempting to terminate unresponsive server process...")
        process.terminate()
        try:
            process.wait(timeout=5)
            print("Unresponsive process terminated.")
        except subprocess.TimeoutExpired:
            print("Unresponsive process did not terminate gracefully, killing it...")
            process.kill()
    else:
        print("The server process terminated or failed to start. Refer to previous error output.")


Attempting to start FastAPI server...
FastAPI server is responsive.
FastAPI server started successfully on port 8000.


In [ ]:
import requests

# It's generally safer to use '127.0.0.1' or 'localhost' for client connections
# to a server running on the same machine, even if the server binds to '0.0.0.0'.
url = "http://127.0.0.1:8000/ask"

payload = {
    "query": "How much is standard delivery?"
}

try:
    response = requests.post(url, json=payload)

    print("Status Code:", response.status_code)
    print("Response:")
    print(response.json())
except requests.exceptions.ConnectionError as e:
    print(f"Connection Error: {e}")
    print("Please ensure the FastAPI server in the previous cell is running correctly.")


Connection Error: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /ask (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7b7665697460>: Failed to establish a new connection: [Errno 111] Connection refused'))
Please ensure the FastAPI server in the previous cell is running correctly.


In [ ]:
process.terminate()

In [ ]:
import requests
import time

payload = {
    "query": "How long do I have to report a damaged item?"
}

max_retries = 5
retry_delay_seconds = 5

response = None
for i in range(max_retries):
    try:
        response = requests.post(url, json=payload, timeout=10) # Added a timeout for the request
        response.raise_for_status() # Raise an exception for HTTP errors
        break # If successful, break the loop
    except requests.exceptions.ConnectionError as e:
        print(f"Connection attempt {i + 1}/{max_retries} failed: {e}")
        if i < max_retries - 1:
            print(f"Retrying in {retry_delay_seconds} seconds...")
            time.sleep(retry_delay_seconds)
        else:
            print("Max retries reached. Could not connect to the server.")
            raise # Re-raise the last exception if all retries fail
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error: {e}")
        print(f"Response content: {response.text if response else 'No response'}")
        raise
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        raise

if response:
    print("Status Code:", response.status_code)
    print(response.json())


Status Code: 200
{'answer': 'I can only answer questions about Zepto policies right now.', 'sources': [], 'confidence': 1.0}


In [ ]:
payload = {
    "query": "What is the capital of India?"
}

response = requests.post(url, json=payload)

print("Status Code:", response.status_code)
print(response.json())

Status Code: 200
{'answer': 'I can only answer questions about Zepto policies right now.', 'sources': [], 'confidence': 1.0}


In [ ]:
response = requests.get("http://127.0.0.1:8000/docs")

print(response.status_code)

200


In [ ]:
{
  "policy_example": {
    "request": {
      "query": "How much is standard delivery?"
    },
    "response": {
      "answer": "Based on the retrieved context: ...",
      "sources": ["doc_01.txt"],
      "confidence": 0.9
    }
  },
  "general_example": {
    "request": {
      "query": "What is the capital of India?"
    },
    "response": {
      "answer": "I can only answer questions about Zepto policies right now.",
      "sources": [],
      "confidence": 1.0
    }
  }
}

{'policy_example': {'request': {'query': 'How much is standard delivery?'},
  'response': {'answer': 'Based on the retrieved context: ...',
   'sources': ['doc_01.txt'],
   'confidence': 0.9}},
 'general_example': {'request': {'query': 'What is the capital of India?'},
  'response': {'answer': 'I can only answer questions about Zepto policies right now.',
   'sources': [],
   'confidence': 1.0}}}

In [ ]:
process.terminate()
print("FastAPI stopped.")

FastAPI stopped.
